In [1]:
import numpy as np
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

def extract_and_process_data(subject_id):
    """
    Extract and process the chest data for a given subject.
    """
    subject_data = wesad_data[subject_id]
    subject_details = subject_data[subject_id]
    chest_data = subject_details['signal']['chest']
    labels = subject_details['label']
    
    # Flatten and reshape each signal to ensure proper concatenation
    ecg = chest_data['ECG'].flatten().reshape(-1, 1)
    emg = chest_data['EMG'].flatten().reshape(-1, 1)
    eda = chest_data['EDA'].flatten().reshape(-1, 1)
    temp = chest_data['Temp'].flatten().reshape(-1, 1)
    resp = chest_data['Resp'].flatten().reshape(-1, 1)
    
    # Combine the features into a single array
    features = np.hstack((ecg, emg, eda, temp, resp))

    return features, labels

def process_all_subjects(subject_ids):
    """
    Process and combine data from all specified subjects.
    """
    all_features = []
    all_labels = []
    
    for subject_id in subject_ids:
        features, labels = extract_and_process_data(subject_id)
        all_features.append(features)
        all_labels.append(labels)
    
    # Combine features and labels from all subjects
    combined_features = np.vstack(all_features)
    combined_labels = np.hstack(all_labels)
    
    return combined_features, combined_labels

# List of all subject IDs in the WESAD dataset
subject_ids = ['S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'S9', 'S10', 'S11', 'S13', 'S14', 'S15', 'S16', 'S17']

# Process data from all subjects
combined_features, combined_labels = process_all_subjects(subject_ids)

# Check the initial label distribution
initial_label_distribution = Counter(combined_labels)
print("Initial label distribution:", initial_label_distribution)

# Remove labels 5, 6, 7
valid_labels = np.isin(combined_labels, [0, 1, 2, 3, 4])
balanced_features = combined_features[valid_labels]
balanced_labels = combined_labels[valid_labels]

# Determine the minimum number of samples for any class
min_samples_per_class = min(Counter(balanced_labels).values())

# Create balanced dataset by randomly sampling min_samples_per_class instances from each class
balanced_features_list = []
balanced_labels_list = []

for label in np.unique(balanced_labels):
    label_indices = np.where(balanced_labels == label)[0]
    if len(label_indices) > min_samples_per_class:
        sampled_indices = np.random.choice(label_indices, min_samples_per_class, replace=False)
    else:
        sampled_indices = label_indices
    balanced_features_list.append(balanced_features[sampled_indices])
    balanced_labels_list.append(balanced_labels[sampled_indices])

# Convert the balanced features and labels to numpy arrays
balanced_features = np.vstack(balanced_features_list)
balanced_labels = np.hstack(balanced_labels_list)

# Check the label distribution after balancing
balanced_label_distribution = Counter(balanced_labels)
print("Balanced label distribution:", balanced_label_distribution)

# Split the balanced data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(balanced_features, balanced_labels, test_size=0.2, random_state=42, stratify=balanced_labels)

# Normalize the features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Initialize and train the classifier
clf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
clf.fit(X_train, y_train)

# Make predictions
y_pred = clf.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

print(f"Accuracy: {accuracy}")
print("Classification Report:")
print(report)

# Check the label distribution in the training set
label_distribution_train = Counter(y_train)
print("Label distribution in the training set:", label_distribution_train)

# Check the label distribution in the test set
label_distribution_test = Counter(y_test)
print("Label distribution in the test set:", label_distribution_test)



KeyboardInterrupt

